# Пример - Спектры изомеров

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **5adcacd**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/5adcacdf23303333a8b03ffc0835da52df4bdb3c.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Девять точных расчётов для гексагональных цепочек. Сравниваем характеристические полиномы двух цепочек из трёх шестиугольников, переносим пять координат на новую границу, восстанавливаем SLR-код и проверяем рекурсии для периодических цепочек.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra 5adcacd](https://github.com/magin67/polyform_algebra/commit/5adcacdf23303333a8b03ffc0835da52df4bdb3c).


## Два изомера из трёх шестиугольников

У обеих цепочек совпадают обычный лесной полином, остовное число и резистивный радиус. Разность характеристических полиномов равна квадрату характеристического полинома $C_5$, а первый различающийся корневой коэффициент имеет грейд 4.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D0%B4%D0%B2%D0%B0-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%B0-%D1%82%D1%80%D1%91%D1%85-%D1%88%D0%B5%D1%81%D1%82%D0%B8%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2#%D1%82%D0%BE%D1%87%D0%BD%D1%8B%D0%B5-%D1%85%D0%B0%D1%80%D0%B0%D0%BA%D1%82%D0%B5%D1%80%D0%B8%D1%81%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-%D0%BF%D0%BE%D0%BB%D0%B8%D0%BD%D0%BE%D0%BC%D1%8B).

In [3]:
from fractions import Fraction
from functools import reduce
from math import gcd, lcm
from string import ascii_lowercase

from sympy import Matrix, Poly, Symbol, diag, expand, eye, factor

from polyform_algebra.boundary_data import spectral_pair_passport, spectral_pair_transfer
from polyform_algebra.core import circumsphere, laplacian, network, spanning_tree_weight
from polyform_algebra.matroid import forest_coefficients
from polyform_algebra.toponomes import laplacian_polynomial


def cycle_edges(vertices):
    return tuple(zip(vertices, vertices[1:] + vertices[:1]))


def unit_graph(vertices, edges):
    return network(tuple(vertices), [(a, b, 1) for a, b in edges])


exits = {"L": 2, "S": 3, "R": 4}


def chain(operations):
    """Attach one hexagon per letter, returning the graph and ordered exit."""
    vertices = tuple("abcdef")
    edges = {tuple(sorted(pair)) for pair in cycle_edges(vertices)}
    active = tuple("ab")
    for letter in operations:
        fresh = tuple(ascii_lowercase[len(vertices):len(vertices)+4])
        assert len(fresh) == 4
        core = (*active[::-1], *fresh)
        vertices += fresh
        edges.update(tuple(sorted(pair)) for pair in cycle_edges(core))
        position = exits[letter]
        active = core[position:position+2]
    return unit_graph(vertices, sorted(edges)), active


def characteristic(graph):
    return Matrix(laplacian(graph).value.values).charpoly(x).as_expr()


x = Symbol("xi")
straight, bent = (chain(letter + "S")[0] for letter in "SL")
chi_straight, chi_bent = map(characteristic, (straight, bent))
pentagon_polynomial = x * (x**2 - 5*x + 5)**2
assert expand(chi_straight - chi_bent - pentagon_polynomial**2) == 0
assert forest_coefficients(straight) == forest_coefficients(bent)
assert laplacian_polynomial(straight)[4] - laplacian_polynomial(bent)[4] == 1
assert all(spanning_tree_weight(graph).value == 204 for graph in (straight, bent))
assert all(circumsphere(graph).value.resistance_radius == Fraction(43595, 41616)
           for graph in (straight, bent))
print("Three-hexagon difference:", factor(chi_straight - chi_bent))

Three-hexagon difference: xi**2*(xi**2 - 5*xi + 5)**4


## Перенос на новую границу

Сравниваем пять перенесённых координат с прямыми минорами для трёх выходов. На отдельном примере проверяем полиномиальную формулу при $\xi=2$, где старый внутренний минор равен нулю.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B9-%D0%BF%D0%B5%D1%80%D0%B5%D0%BD%D0%BE%D1%81-%D0%BF%D1%80%D0%B8-%D1%81%D0%BC%D0%B5%D0%BD%D0%B5-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D1%8B#%D1%82%D1%80%D0%B8-%D0%B2%D1%8B%D1%85%D0%BE%D0%B4%D0%B0-%D0%B5%D0%B4%D0%B8%D0%BD%D0%B8%D1%87%D0%BD%D0%BE%D0%B3%D0%BE-%D1%88%D0%B5%D1%81%D1%82%D0%B8%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%B0).

In [4]:
kernel = unit_graph("abcdef", cycle_edges(tuple("abcdef")))
entry = tuple("ab")
swap = eye(5)
swap.row_swap(1, 2)
transfers = {
    letter: spectral_pair_transfer(kernel, entry, tuple("abcdef"[i:i+2]), x, shared_conductance=1)
    for letter, i in exits.items()
}
operators = {letter: matrix * swap for letter, matrix in transfers.items()}
seed = Matrix(spectral_pair_passport(kernel, entry, x))


def spectral_state(operations):
    result = seed
    for letter in operations:
        result = (operators[letter] * result).applyfunc(expand)
    return result


for operations in ("L", "S", "R", "LS", "RS"):
    graph, active = chain(operations)
    direct_state = Matrix(spectral_pair_passport(graph, active, x))
    assert spectral_state(operations) == direct_state

# A separate attachment at a zero of the old interior minor.
old = unit_graph("acb", (("a", "c"), ("c", "b")))
runner = unit_graph("adeb", (("a", "d"), ("d", "e"), ("e", "b")))
glued = network(tuple("acbde"), old.edges + runner.edges)
old_state = Matrix(spectral_pair_passport(old, entry, x))
exit_pair = tuple("de")
transfer = spectral_pair_transfer(runner, entry, exit_pair, x)
assert old_state[4].subs(x, 2) == 0
assert (transfer * old_state).applyfunc(expand) == Matrix(spectral_pair_passport(glued, exit_pair, x))
print("Five-coordinate transfer checked, including xi = 2")

Five-coordinate transfer checked, including xi = 2


## Симметрии SLR-кода

Строим орбиту одного кода по отражению и обращению. Сверяем симметрии матриц переноса и совпадение спектральных полиномов в этой орбите.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/slr-%D0%BA%D0%BE%D0%B4-%D0%B3%D0%B5%D0%BA%D1%81%D0%B0%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D1%85-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%B5%D0%BA#%D0%BE%D1%82%D1%80%D0%B0%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B8-%D0%BE%D0%B1%D1%80%D0%B0%D1%89%D0%B5%D0%BD%D0%B8%D0%B5-%D1%86%D0%B5%D0%BF%D0%B8).

In [5]:
mirror_table = str.maketrans("LR", "RL")


def orbit(word):
    reflected = word.translate(mirror_table)
    return {word, reflected, word[::-1], reflected[::-1]}


assert transfers["R"] == swap * transfers["L"] * swap
assert transfers["S"] == swap * transfers["S"] * swap
turn_word = "LS"
equivalent_codes = sorted("S" + word for word in orbit(turn_word))
assert equivalent_codes == ["SLS", "SRS", "SSL", "SSR"]
assert len({spectral_state(code[1:] + "S")[0] for code in equivalent_codes}) == 1
print("Equivalent SLR codes:", ", ".join(equivalent_codes))

Equivalent SLR codes: SLS, SRS, SSL, SSR


## Две точные спектральные разности

Сравниваем полиномы для слов $S,L$ и $LL,LR$. Здесь и ниже слова в коде задают внутренние повороты, а полное обозначение цепочки начинается с $S$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5-%D1%80%D0%B0%D0%B7%D0%BD%D0%BE%D1%81%D1%82%D0%B8-slr-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%B5%D0%BA#%D1%81%D0%BD%D0%B0%D1%87%D0%B0%D0%BB%D0%B0-%D0%B4%D0%B2%D0%B5-%D0%BA%D0%BE%D0%BD%D0%BA%D1%80%D0%B5%D1%82%D0%BD%D1%8B%D0%B5-%D1%80%D0%B0%D0%B7%D0%BD%D0%BE%D1%81%D1%82%D0%B8).

In [6]:
for first, second, expected in (
    ("S", "L", pentagon_polynomial**2),
    ("LL", "LR", -(x-2)**2 * pentagon_polynomial**2),
):
    difference = spectral_state(first + "S")[0] - spectral_state(second + "S")[0]
    assert expand(difference - expected) == 0
    print("Spectral difference", first, second, "=", factor(difference))
delta = operators["S"] - operators["L"]
assert delta.subs(x, -1).rank() == 2

Spectral difference S L = xi**2*(xi**2 - 5*xi + 5)**4


Spectral difference LL LR = -xi**2*(xi - 2)**2*(xi**2 - 5*xi + 5)**4


## Декодирование корневого состояния

Специализируем спектральный паспорт в $\xi=-1$. Сравниваем с прямыми минорами и снимаем шаги переноса в обратном порядке по знакам двух линейных форм.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D0%BA%D0%BE%D1%80%D0%BD%D0%B5%D0%B2%D1%8B%D0%B5-%D0%BB%D0%B5%D1%81%D0%B0-%D0%B8-%D0%B4%D0%B5%D0%BA%D0%BE%D0%B4%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D0%BE%D0%B3%D0%BE-%D1%81%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D0%BD%D0%B8%D1%8F#%D0%BE%D0%B1%D1%80%D0%B0%D1%82%D0%BD%D1%8B%D0%B9-%D0%B0%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC-%D0%B8-%D1%82%D0%B5%D0%BE%D1%80%D0%B5%D0%BC%D0%B0-%D0%B4%D0%B5%D0%BA%D0%BE%D0%B4%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F).

In [7]:
sign = diag(1, -1, -1, -1, 1)
integer_operators = {letter: sign * matrix.subs(x, -1) * sign
                     for letter, matrix in operators.items()}
integer_seed = sign * seed.subs(x, -1)
assert tuple(integer_seed) == (320, 144, 144, 56, 55)


def rooted_state(operations):
    result = integer_seed
    for letter in operations:
        result = integer_operators[letter] * result
    return result


def decode_operations(value):
    inverse = {letter: matrix.inv() for letter, matrix in integer_operators.items()}
    decoded = []
    while value != integer_seed:
        a, b, c, d, e = value
        assert a*e - b*c + d*d == 0
        sectors = {(True, True): "S", (False, True): "L", (True, False): "R"}
        assert 9*b != 4*a and 9*c != 4*a
        letter = sectors[(9*b > 4*a, 9*c > 4*a)]
        predecessor = inverse[letter] * value
        assert 0 < predecessor[0] < a
        decoded.append(letter)
        value = predecessor
    return "".join(reversed(decoded))


for operations in ("S", "LS", "RLS", "SLRS"):
    value = rooted_state(operations)
    graph, active = chain(operations)
    assert value == sign * Matrix(spectral_pair_passport(graph, active, -1))
    assert value[0] == sum(laplacian_polynomial(graph))
    assert decode_operations(value) == operations
print("Rooted-state decoding:", decode_operations(rooted_state("SLRS")))

Rooted-state decoding: SLRS


## Восстановление по граничному отклику

Из блока $(I+L)^{-1}$ на выбранной границе восстанавливаем примитивное целое состояние и проверяем полученный код. Расчёт использует точные рациональные значения.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D0%B2%D0%BE%D1%81%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%BA%D0%B8-%D0%BF%D0%BE-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%BD%D0%BE%D0%BC%D1%83-%D0%BE%D1%82%D0%BA%D0%BB%D0%B8%D0%BA%D1%83#%D0%BF%D0%BE%D1%87%D0%B5%D0%BC%D1%83-%D0%BD%D0%BE%D1%80%D0%BC%D0%B8%D1%80%D0%BE%D0%B2%D0%BA%D0%B0-%D0%BD%D0%B5-%D1%82%D0%B5%D1%80%D1%8F%D0%B5%D1%82-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%BA%D1%83).

In [8]:
operations = "LRSS"
graph, active = chain(operations)
matrix = eye(len(graph.context.labels)) + Matrix(laplacian(graph).value.values)
positions = [graph.context.labels.index(label) for label in active]
boundary_inverse = matrix.inv().extract(positions, positions)
r, s, h = map(Fraction, (boundary_inverse[0, 0], boundary_inverse[1, 1], boundary_inverse[0, 1]))
projective = (Fraction(1), r, s, h, r*s-h*h)
scale = lcm(*(part.denominator for part in projective))
recovered = Matrix([int(scale * part) for part in projective])
assert reduce(gcd, recovered) == 1 and recovered == rooted_state(operations)
assert decode_operations(recovered) == operations
print("Code recovered from exact boundary response:", "S" + operations[:-1])

Code recovered from exact boundary response: SLRS


## Постоянный прямой поворот

Проверяем характеристический полином матрицы прямого шага и остовную рекурсию для первых пяти цепочек.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D0%BF%D0%BB%D0%B5%D1%82%D0%B5%D0%BD%D0%B8%D0%B5-slr-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%B5%D0%BA-%D0%B8-%D0%BF%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D0%BD%D0%BD%D1%8B%D0%B9-%D0%BF%D0%BE%D0%B2%D0%BE%D1%80%D0%BE%D1%82#%D0%BF%D1%8F%D1%82%D1%8C-%D0%BA%D0%BE%D0%BE%D1%80%D0%B4%D0%B8%D0%BD%D0%B0%D1%82-%D0%B8-%D1%81%D1%82%D0%B5%D0%BF%D0%B5%D0%BD%D1%8C-%D0%BE%D0%B4%D0%BD%D0%BE%D0%B9-%D0%BC%D0%B0%D1%82%D1%80%D0%B8%D1%86%D1%8B).

In [9]:
q = Symbol("q")
straight_operator = operators["S"]
alpha = (x**2-6*x+6) * (x**2-4*x+2)
beta = 2*(x-1)*(x**3-9*x**2+25*x-19)
quartic = q**4 - alpha*q**3 + beta*q**2 - alpha*q + 1
assert expand(straight_operator.charpoly(q).as_expr() - (q-1)*quartic) == 0
tree_counts = [spanning_tree_weight(chain("S"*n)[0]).value for n in range(5)]
assert tree_counts == [6, 35, 204, 1189, 6930]
assert all(c == 6*b - a for a, b, c in zip(tree_counts, tree_counts[1:], tree_counts[2:]))
print("Constant straight turns, spanning trees:", tuple(map(str, tree_counts)))

Constant straight turns, spanning trees:

 ('6', '35', '204', '1189', '6930')


## Первые различающиеся грейды

Прямо вычисляем характеристические полиномы и сравниваем первые различающиеся грейды. Отдельно проверяем формулы для четвёртого и шестого моментов через число изгибов и соседних одинаковых поворотов.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D0%B0%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F-%D1%80%D0%B0%D0%B7%D0%BB%D0%B8%D1%87%D0%B8%D0%BC%D0%BE%D1%81%D1%82%D1%8C-slr-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%B5%D0%BA#%D1%81%D0%BE%D0%B2%D0%BF%D0%B0%D0%B4%D0%B0%D1%8E%D1%89%D0%B8%D0%B5-%D0%BD%D0%B0%D1%87%D0%B0%D0%BB%D1%8C%D0%BD%D1%8B%D0%B5-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D1%8B).

In [10]:
for first, second, expected_grade in (("LL", "LR", 6), ("LSL", "LRL", 4), ("SL", "LS", None)):
    first_graph = chain(first + "S")[0]
    second_graph = chain(second + "S")[0]
    difference = Poly(characteristic(first_graph) - characteristic(second_graph), x)
    grade = None if difference.is_zero else len(first_graph.context.labels) - difference.degree()
    assert grade == expected_grade
    print("First different grade", first, second, "=", grade)
for word in ("LL", "LR", "LSL", "LRL"):
    graph = chain(word + "S")[0]
    matrix = Matrix(laplacian(graph).value.values)
    n = len(word) + 2
    bends = sum(letter != "S" for letter in word)
    repeated_turns = sum(a == b and a != "S" for a, b in zip(word, word[1:]))
    assert (matrix**4).trace() == 648*n - 228 + 4*bends
    assert (matrix**6).trace() == 13656*n - 8112 + 576*bends + 6*repeated_turns

First different grade LL LR = 6
First different grade LSL LRL = 4
First different grade SL LS = None


## Перенос для периодического слова

Для периодов $LR$, $SL$, $LLS$ при $\xi=-1$ проверяем подобие циклических фаз, взаимно обратный множитель четвёртой степени и скалярную рекурсию с постоянной правой частью. Первые значения сверяем с прямым корневым полиномом.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B-%D0%B8%D0%B7%D0%BE%D0%BC%D0%B5%D1%80%D0%BE%D0%B2/%D0%BF%D0%B5%D1%80%D0%B8%D0%BE%D0%B4%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5-slr-%D1%86%D0%B5%D0%BF%D0%BE%D1%87%D0%BA%D0%B8---%D1%80%D0%B5%D0%BA%D1%83%D1%80%D1%81%D0%B8%D0%B8-%D0%B8-%D1%81%D0%BF%D0%B5%D0%BA%D1%82%D1%80%D1%8B#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D1%80%D0%B0%D1%81%D1%87%D1%91%D1%82%D0%B0-%D0%B4%D0%BB%D1%8F-%D0%BF%D0%B5%D1%80%D0%B8%D0%BE%D0%B4%D0%B0-lr).

In [11]:
def period_matrix(word):
    matrix = eye(5)
    for letter in word:
        matrix = integer_operators[letter] * matrix
    return matrix


for word in ("LR", "SL", "LLS"):
    matrix = period_matrix(word)
    rotated = period_matrix(word[1:] + word[0])
    first = integer_operators[word[0]]
    assert rotated * first == first * matrix
    characteristic_period = matrix.charpoly(q).as_expr()
    quotient, remainder = Poly(characteristic_period, q).div(Poly(q-1, q))
    coefficients = quotient.all_coeffs()
    assert remainder.is_zero and coefficients == coefficients[::-1]
    row = integer_operators["S"][0, :]
    sequence = [(row * matrix**k * integer_seed)[0] for k in range(9)]
    residuals = [sum(a*b for a, b in zip(coefficients, reversed(sequence[k:k+5]), strict=True))
                 for k in range(5)]
    assert len(set(residuals)) == 1
    assert Matrix(5, 5, lambda i, j: sequence[i+j]).det() != 0
    for repeats in range(2):
        graph = chain(word*repeats + "S")[0]
        assert sequence[repeats] == sum(laplacian_polynomial(graph))
    print("Period", word, "first rooted totals:", sequence[:3])

Period LR first rooted totals: [28435, 220480560, 1709242570740]
Period SL first rooted totals: [28435, 221771751, 1729329809694]
Period LLS first rooted totals: [28435, 19514594869, 13389935419018422]
